On s'occupe des tables des constructeurs

In [1]:
import numpy as np
import pandas as pd

from src import import_data as i_d
#importation des données

i_d.charger_donnees_depuis_bureau()


#constructor_standing = pd.read_csv("donnees_formule_un/constructor_standings.csv")
#st_res = pd.read_csv("donnees_formule_un/constructor_results.csv")

Dossier trouvé : /Users/gabriels./Desktop/donnees_formule_un
Fichiers CSV trouvés : [PosixPath('/Users/gabriels./Desktop/donnees_formule_un/circuits.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/status.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/lap_times.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/sprint_results.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/drivers.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/races.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructors.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructor_standings.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/qualifying.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/driver_standings.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructor_results.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/pit_stops.csv'), PosixPath('/Users/gabri

structure de la table constructors :
- constructorID = int
- constructorRef = object
- name = object
- nationality = object
- url = object

structure de la table constructor_results:
- constructorResultsID = int
- raceID = int 
- constructorID = int
- points = float
- status = object

structure de la table constructor_standings:
- constructorStandingsID = int
- raceID = int
- constructorID = int
- points = float
- positionText = object
- wins = int

In [2]:
#recherche des Na : on a des \\N = équivalent
constructors
constructors[constructors.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

constructor_standings
constructor_standings[constructor_standings.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

constructor_results
constructor_results[constructor_results.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
# un \\N à chaque ligne


#on remplace les \N par des NA:
constructor_results.replace("\\N", np.nan, inplace= True)


races.replace("\\N", np.nan, inplace= True)

**Question : quelle écurie a gagné le plus de courses ? 
Faire un classement des écuries selon le nombre de victoires cumulées.**

In [3]:
#on fait un groubpy pour récupérer le constructorId ayant remporté le plus de points pour chaque course
max_course = constructor_results.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
#la colonne constructorId n'est pas affiché : on va merge avec d'autres tables pour récupérer les id puis les noms des constructeurs
max_course = pd.merge(max_course, races, on = ["raceId"])
max_course =  max_course[["raceId", "premier"]]
max_course
gagnants = pd.merge(
    max_course,
    constructor_results,
    left_on=["raceId", "premier"],
    right_on=["raceId", "points"], #on fait correspondre "premier" et "points"
    how="inner"
)
gagnants = gagnants[["raceId", "constructorId", "points"]]
gagnants = pd.merge(gagnants, constructors, on = ["constructorId"], how = "inner")
gagnants = gagnants[["raceId", "name", "points"]]
gagnants = gagnants.groupby("name", as_index= False).size()
gagnants.sort_values(by = "size", ascending= False)


,name,size
14,Ferrari,239
25,McLaren,192
42,Williams,126
27,Mercedes,117
31,Red Bull,111
36,Team Lotus,46
32,Renault,34
4,Benetton,28
5,Brabham,24
21,Lotus-Climax,22


La table ci-dessus indique que le contsructeur qui a cumulé le plus de courses remportées d'après la table race est ferrari avec 239 victoires 

**Question 2: quel constructeur a remporté le plus de saisons ? Faire un classement.**

In [4]:
#on veut récupérer la course la plus ancienne de la table race
races["date"].min()
#la première course de la base de donnée a été effectuée le 13/05/1950
season = seasons
season #on remarque que les saisons sont découpées par années et qu'il n'y a pas de "débordement" d'une année à l'autre
#l'écurie qui aura remporté le plus de points dans la saison la remporte
#on veut regrouper à la fois par année et par constructeur

total_course = pd.merge(races, constructor_results, how = "inner")
total_course = total_course.loc[:, ["raceId", "year", "constructorId", "points"]]
total_course

,raceId,year,constructorId,points
0,1,2009,23,18.0
1,1,2009,1,0.0
2,1,2009,7,11.0
3,1,2009,4,4.0
4,1,2009,3,3.0
...,...,...,...,...
12500,1132,2024,117,10.0
12501,1132,2024,3,2.0
12502,1132,2024,215,1.0
12503,1132,2024,15,0.0


In [5]:
total_course
races.groupby("year")["raceId"].count()
#on peut faire le groupby:

tbl = constructor_results.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
tbl = pd.merge(constructor_results, tbl, left_on = ["raceId", "points"], right_on= ["raceId", "premier"])
tbl


,constructorResultsId,raceId,constructorId,points,status,premier
0,1,18,1,14.0,NaN,14.0
1,13,19,2,11.0,NaN,11.0
2,23,20,6,18.0,NaN,18.0
3,34,21,6,18.0,NaN,18.0
4,45,22,6,16.0,NaN,16.0
...,...,...,...,...,...,...
1087,16971,1129,1,28.0,NaN,28.0
1088,16972,1129,131,28.0,NaN,28.0
1089,16980,1130,9,29.0,NaN,29.0
1090,16990,1131,131,45.0,NaN,45.0


In [6]:

tbl2 = pd.merge(races, tbl, on = "raceId")
tbl2.columns
tbl2 = tbl2[["year", "raceId", "constructorId", "name", "points"]]
tbl2


,year,raceId,constructorId,name,points
0,2009,1,23,Australian Grand Prix,18.0
1,2009,2,23,Malaysian Grand Prix,7.0
2,2009,3,9,Chinese Grand Prix,18.0
3,2009,4,23,Bahrain Grand Prix,14.0
4,2009,5,23,Spanish Grand Prix,18.0
...,...,...,...,...,...
1087,2024,1129,1,Canadian Grand Prix,28.0
1088,2024,1129,131,Canadian Grand Prix,28.0
1089,2024,1130,9,Spanish Grand Prix,29.0
1090,2024,1131,131,Austrian Grand Prix,45.0


In [7]:
tbl2 = tbl2.groupby("year", as_index= False)["constructorId"].value_counts()
tbl2

,year,constructorId,count
0,1958,118,5
1,1958,6,3
2,1958,87,2
3,1959,170,5
4,1959,6,2
...,...,...,...
261,2023,131,1
262,2024,9,6
263,2024,1,3
264,2024,6,2


In [8]:
count_max = tbl2.groupby("year", as_index= False)["count"].max()
count_max
merged = pd.merge(count_max, tbl2, on = ["year"])
merged = merged[merged["count_x"] == merged["count_y"]]
merged = merged[["year", "count_x", "constructorId"]]
merged

,year,count_x,constructorId
0,1958,5,118
3,1959,5,170
7,1960,6,170
11,1961,5,6
13,1962,4,66
...,...,...,...
246,2020,11,131
251,2021,11,9
255,2022,15,9
258,2023,18,9


In [9]:
gagnants_saisons = pd.merge(merged, constructors, on = "constructorId")
total_win_saisons = gagnants_saisons.groupby("name", as_index= False)["year"].size()
total_win_saisons.sort_values(by = "size", ascending= False)


,name,size
4,Ferrari,13
8,McLaren,11
10,Red Bull,9
15,Williams,8
9,Mercedes,7
12,Team Lotus,5
5,Lotus-Climax,3
1,Benetton,2
2,Brabham-Repco,2
3,Cooper-Climax,2


On en conclut que Ferrari cumule le plus de saisons remportées. 

**Question python base : calculons un test de khi-deux entre la variable de position au départ et la variable de classement**


In [10]:

# Données chargées dans la variable : status
# Données chargées dans la variable : sprint_results
# Données chargées dans la variable : drivers
# Données chargées dans la variable : races
# Données chargées dans la variable : constructors
# Données chargées dans la variable : constructor_standings
# Données chargées dans la variable : qualifying
# Données chargées dans la variable : driver_standings
# Données chargées dans la variable : constructor_results
# Données chargées dans la variable : pit_stops
# Données chargées dans la variable : seasons
# Données chargées dans la variable : results


In [11]:
results
#on s'intéresse aux variables grid et position


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26514,26520,1132,839,214,31,18,16,16,16,0.0,50,\N,\N,46,16,1:30.875,233.371,12
26515,26521,1132,815,9,11,0,17,17,17,0.0,50,\N,\N,50,6,1:29.707,236.409,12
26516,26522,1132,855,15,24,14,18,18,18,0.0,50,\N,\N,43,17,1:31.014,233.014,12
26517,26523,1132,847,131,63,1,\N,R,19,0.0,33,\N,\N,3,19,1:31.298,232.289,34


Question préliminaire: y a-t-il un nombre constant de places au départ d'une course ?

In [12]:
import csv
grid = []
position = []
with open('/Users/gabriels./Desktop/donnees_formule_un/sprint_results.csv', newline = '', encoding = 'utf-8') as results:
    table = csv.DictReader(results)
    for row in table:
        # on veut traiter les \NA: on les remplace par 0 dans les variables
        # 'grid' et 'position'
        # on stocke les données dans 2 listes grid et position.
        print(row['grid'], row['position'])
        row['grid'] = row['grid'].replace('\\N', '0')
        row['grid'] = int(row['grid'])
        row['position'] = row['position'].replace('\\N', '0')
        row['position'] = int(row['position'])
        grid.append(row['grid'])
        position.append(row['position'])


print(position)
print(grid)

#on remarque que la plupart des courses ont 19 places, mais pas toutes.


2 1
1 2
3 3
4 4
6 5
7 6
11 7
10 8
8 9
13 10
9 11
12 12
17 13
15 14
14 15
16 16
18 17
19 18
20 19
5 \N
1 1
3 2
5 3
4 4
2 5
8 6
7 7
10 8
9 9
12 10
13 11
11 12
14 13
16 14
15 15
17 16
20 17
19 18
18 19
6 \N
2 1
1 2
5 3
3 4
20 5
7 6
6 7
4 8
10 9
11 10
8 11
9 12
14 13
15 14
12 15
16 16
17 17
13 18
18 19
19 20
1 1
2 2
7 3
10 4
3 5
6 6
8 7
4 8
5 9
12 10
11 11
16 12
9 13
13 14
15 15
19 16
17 17
20 18
18 19
0 \N
1 1
2 2
3 3
4 4
13 5
5 6
6 7
9 8
7 9
12 10
15 11
16 12
17 13
18 14
10 15
11 16
14 17
19 18
20 19
8 \N
3 1
5 2
8 3
2 4
9 5
10 6
4 7
1 8
13 9
12 10
14 11
20 12
17 13
18 14
19 15
15 16
6 17
7 18
16 19
11 \N
2 1
1 2
3 3
4 4
5 5
8 6
6 7
9 8
7 9
11 10
13 11
14 12
17 13
18 14
12 15
15 16
10 17
0 18
16 \N
0 \N
1 1
2 2
5 3
7 4
6 5
4 6
8 7
15 8
3 9
18 10
17 11
9 12
11 13
10 14
12 15
13 16
14 17
20 18
16 19
0 20
1 1
2 2
6 3
3 4
4 5
5 6
7 7
10 8
9 9
11 10
14 11
12 12
17 13
18 14
19 15
13 16
20 17
16 18
8 \N
15 \N
1 1
3 2
2 3
4 4
12 5
5 6
17 7
9 8
11 9
13 10
18 11
6 12
16 13
19 14
15 15
7 \N
10 \N
8

In [23]:
# on veut isoler les différentes sous-séquences dans la liste grid
# on sait que la liste grid est déjà ordonnée de manière croissante par courses
Liste = []
i = 0
j = 0
# while j < len(grid):
        #Liste.append([])
        #while grid[j +1] > grid[j]:
                #Liste[i].append(grid[j])
        #i += 1



def function1():
    list2 = []
    for i in range(len(grid)):
         if i < len(grid) - 2:
              if grid[i] > grid[i+1]:
                  list2.append(i)
    return list2

print(function1())

grid[0:6]

[0, 6, 7, 9, 12, 13, 18, 19, 22, 23, 25, 27, 30, 33, 36, 37, 38, 39, 40, 42, 44, 45, 46, 49, 53, 56, 59, 63, 66, 69, 71, 75, 77, 78, 84, 87, 93, 98, 99, 102, 105, 106, 108, 111, 114, 115, 118, 119, 120, 125, 127, 133, 135, 136, 138, 143, 144, 147, 149, 150, 152, 157, 158, 162, 167, 170, 174, 176, 177, 179, 181, 184, 186, 190, 193, 194, 196, 198, 199, 201, 204, 207, 208, 210, 213, 215, 218, 219, 220, 224, 225, 227, 231, 234, 236, 239, 240, 243, 244, 247, 250, 252, 255, 257, 258, 259, 269, 270, 272, 273, 274, 275, 277, 279, 281, 286, 289, 293, 295, 297]


[2, 1, 3, 4, 6, 7]